# S4 — 時間序列 + EDA 實戰

> **時間**：2 小時  
> **資料**：`orders_enriched.csv`（S3 產出）  
> **學完能幹嘛**：回答老闆「這幾個月業績怎麼樣？」「下個月該不該進貨？」「哪個品類相關性最強？」

---

## 上節回顧 + 本節為什麼重要

S3 我們合併了三表、算出各地區 Top 商品。但真實的商業問題都有**時間維度**：

- 哪個**月**營收最高？
- 過去 7 天的**趨勢**是往上還是往下？
- 每個**星期幾**的銷售有差嗎？
- 品類之間的銷售有**相關性**嗎？

這些都是**時間序列分析 (Time Series)** 和 **探索式分析 (EDA)** 的核心問題。

> **DA 視角**：EDA 是所有資料分析的第一步，目的是「先看懂資料、再做決策」。


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    '../datasets/ecommerce/orders_enriched.csv',
    parse_dates=['order_date'],
)
print('資料形狀:', df.shape)
print('日期範圍:', df['order_date'].min(), '~', df['order_date'].max())
df.head(3)

---
## 核心觀念 1／3 — 日期操作：`to_datetime` 與 `.dt` accessor

一旦把欄位轉成 datetime 型別，就能用 `.dt` 快速拆出年、月、日、星期。


In [ ]:
# 從 order_date 拆出常用的時間欄位
df['year']     = df['order_date'].dt.year
df['month']    = df['order_date'].dt.month
df['weekday']  = df['order_date'].dt.day_name()   # Monday, Tuesday, ...
df['year_mon'] = df['order_date'].dt.to_period('M')  # 2025-01, 2025-02, ...

df[['order_date', 'year', 'month', 'weekday', 'year_mon']].head()

**口訣**：`.dt.年/月/日/星期/週數/季度`，像 Excel 的 `YEAR() MONTH()` 一樣好用。


---
## 核心觀念 2／3 — `resample`：時間重新取樣

`resample` 是專門給時序資料做「重新切粒度」的工具。例如：
- 原本每筆訂單 → 重取樣成「每日總和」「每月總和」「每週平均」

**前提**：index 必須是 datetime。


In [ ]:
# 把 order_date 設為 index
ts = df.set_index('order_date').sort_index()

# 月度營收（'ME' = Month End）
monthly = ts['amount'].resample('ME').sum()
print('📅 月度營收:')
print(monthly)

In [ ]:
# 每週訂單數（'W' = Week）
weekly_count = ts['order_id'].resample('W').count()
print('📅 每週訂單數（前 8 週）:')
print(weekly_count.head(8))

**常用 resample 粒度**：`'D'` 日、`'W'` 週、`'ME'` 月底、`'QE'` 季末、`'YE'` 年底。


---
## 核心觀念 3／3 — `rolling`：移動平均（平滑雜訊）

日資料常有大起大落的雜訊，**移動平均**能平滑趨勢、看出大方向。


In [ ]:
daily = ts['amount'].resample('D').sum()        # 每日營收
ma7   = daily.rolling(window=7).mean()          # 7 日移動平均

compare = pd.DataFrame({
    '每日原值': daily,
    'MA7':     ma7,
}).dropna()

print('📈 7 日移動平均（前 10 天）:')
print(compare.head(10).round(0))

**口訣**：`.rolling(window=N).mean()` → 滑動窗口平均，N 越大越平滑。


---
## EDA 三大招：describe、corr、value_counts

拿到任何資料，花 5 分鐘做這三件事就能對資料有基本認識。


> ### 💡 知識補給站 — EDA 系統化清單
> 
> EDA 不是隨便看看，有一份**系統化清單**可以確保不遺漏：
> 
> | 步驟 | 做什麼 | 工具 |
> |---|---|---|
> | 1. 形狀與型別 | 幾列幾欄？型別對嗎？ | `shape`, `dtypes`, `info()` |
> | 2. 缺值地圖 | 哪些欄位缺最多？ | `isna().sum()`, `isna().mean()` |
> | 3. 數值摘要 | min/max 合理嗎？有沒有離群值？ | `describe()` |
> | 4. 類別分布 | 有沒有拼寫錯誤或稀有類別？ | `value_counts()` |
> | 5. 相關性 | 哪些欄位高度相關？ | `corr()`（但記得：相關 ≠ 因果） |
> | 6. 時間趨勢 | 有沒有明顯的季節性或異常跳動？ | `resample().sum()` + 畫圖 |
> 
> 把這 6 步跑完，你就對資料有 **80% 的掌握**。
> 
> 延伸工具：`ydata-profiling`（原 pandas-profiling）— 一行指令就能產出完整 EDA 報告。
> 
> *延伸關鍵字：EDA checklist, exploratory data analysis, ydata-profiling, systematic analysis*

In [ ]:
# 招 1: describe — 數值欄的統計摘要
print('📊 數值摘要:')
print(df[['qty', 'amount', 'unit_price', 'stock_qty']].describe().round(1))

In [ ]:
# 招 2: value_counts — 類別欄的分布
print('🏷️  地區分布:')
print(df['region'].value_counts())
print('\n🏷️  品類分布:')
print(df['category'].value_counts())

In [ ]:
# 招 3: 相關係數矩陣
num_cols = ['qty', 'amount', 'unit_price', 'stock_qty']
print('🔗 相關係數:')
print(df[num_cols].corr().round(2))

**解讀相關係數**：
- `1.0` 完美正相關 | `0` 無關 | `-1.0` 完美負相關
- `|r| > 0.7` 強相關 | `0.3~0.7` 中 | `< 0.3` 弱


> ### 💡 知識補給站 — Correlation ≠ Causation（相關不等於因果）
> 
> 剛才看到 `unit_price` 和 `amount` 的相關係數很高 — 但這**不代表**「提高單價就能增加營收」。
> 
> **經典反例**：冰淇淋銷量和溺水人數高度正相關 → 難道吃冰淇淋會溺水？不是！背後有**共同的隱藏變數 (confounding variable)**：夏天。天氣熱 → 買冰淇淋的人多 + 游泳的人多。
> 
> **在報告中**：
> - ✅ 安全的寫法：「A 與 B 呈正相關 (r = 0.85)」
> - ❌ 危險的寫法：「A **導致** B 增加」← 這需要**實驗**（如 A/B test）或進階統計方法才能證明
> 
> **因果推論**需要：控制實驗（A/B test）、工具變數（IV）、自然實驗、或 RCT — 這些是統計學和 ML 的進階主題。
> 
> *延伸關鍵字：correlation vs causation, confounding variable, A/B test, causal inference*

---
## 實務 Case — 銷售趨勢與洞察

**情境**：行銷總監要你在下次會議前提出三個發現：
1. 月度營收走勢 → **最旺的月份**是哪個？
2. 星期幾的銷售最好？
3. 哪個品類 × 地區組合最值得重點經營？


In [ ]:
# Q1: 月度趨勢
monthly_rev = df.groupby('year_mon')['amount'].sum()
best = monthly_rev.idxmax()
print(f'📈 月度營收:')
print(monthly_rev)
print(f'\n🏆 最旺月份: {best} (NT$ {monthly_rev.max():,.0f})')

In [ ]:
# Q2: 星期幾表現（並依自然順序排序）
week_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
weekday_rev = df.groupby('weekday')['amount'].sum().reindex(week_order)
print('📅 星期幾營收:')
print(weekday_rev)
print(f'\n🏆 最強: {weekday_rev.idxmax()}  🐌 最弱: {weekday_rev.idxmin()}')

In [ ]:
# Q3: 地區 × 品類熱點矩陣
hot = df.pivot_table(
    index='category', columns='region', values='amount', aggfunc='sum', fill_value=0,
)
print('🔥 地區 × 品類 營收熱點:')
print(hot)

# 找出 Top 1 組合
stacked = hot.stack().sort_values(ascending=False)
print(f'\n🏆 最強組合: {stacked.index[0]} = NT$ {stacked.iloc[0]:,.0f}')

> ### 💡 知識補給站 — 當心 Simpson's Paradox
> 
> 我們剛才看到「整體最強的品類 × 地區組合」，但如果**拆開不同時間段**看，結論可能完全反轉 — 這就是 **Simpson's Paradox**（辛普森悖論）。
> 
> **範例情境**：
> - 整體：A 品類營收 > B 品類 ✓
> - 但按季度拆開：**每一季 B 都贏 A** → 只是 A 在大季度有更多訂單，拉高了整體數字
> 
> **解法**：做分析時**多加一層 breakdown**（按時間、按地區、按客群），看看結論是否一致。
> 
> 口訣：「**整體趨勢 ≠ 分組趨勢，多切一刀才安心**」。
> 
> 這在醫學研究、A/B testing、商業分析中都是經典陷阱 — 聚合數據可以騙人。
> 
> *延伸關鍵字：Simpson's Paradox, ecological fallacy, stratified analysis, confounding variable*

In [ ]:
# 把分析結果存起來，S5/S6 視覺化直接拿去畫
monthly_rev.to_csv('../datasets/ecommerce/monthly_revenue.csv', header=['amount'])
print('已存 monthly_revenue.csv')

---
## 課堂練習（40 min）

### 🟢 送分題
1. 用 `df` 算出每個「月份 (1~12)」的平均訂單金額
2. 列出訂單數最多的前 3 天（日期）


In [ ]:
# TODO


### 🟡 核心題
1. 計算**每月訂單數**的 3 個月移動平均
2. 找出 **2025 年第 4 季 (10~12 月)** 的總營收
3. 算出每個品類的訂單金額**中位數**並排序


In [ ]:
# TODO


### 🔴 挑戰題
做一份「月度經營報告」，每月一列，包含：
- 總訂單數
- 總營收
- 活躍顧客數（`nunique`）
- 客單價（= 總營收 / 總訂單數）
- 與上月的營收成長率 (%)（提示：`.pct_change()`）

> 這張表就是真實 DA 每月都要交的**月報表**範本。


In [ ]:
# TODO


---
## 小結與速查表

| 想做什麼 | 怎麼寫 |
|---|---|
| 轉日期 | `pd.to_datetime(col)` |
| 取年月日 | `.dt.year / .dt.month / .dt.day_name()` |
| 轉年月字串 | `.dt.to_period('M')` |
| 每日/週/月聚合 | `.resample('D'/'W'/'ME').sum()` |
| 移動平均 | `.rolling(window=7).mean()` |
| 成長率 | `.pct_change()` |
| 數值摘要 | `df.describe()` |
| 類別分布 | `df['x'].value_counts()` |
| 相關係數 | `df[num_cols].corr()` |
| Unique 計數 | `.nunique()` |

**下節預告 S5**：我們已經有了一大堆洞察數字，但**「數字不會說話，圖會」**。S5 教你 5 種必懂的視覺化圖，把今天算出來的結果變成一份能交給老闆的簡報。
